# Synthetic spatial-expression simulation

This notebook uses scDesign3 to fit spatial expression patterns in reference
AnnData datasets and simulate count matrices across a spatial-signal gradient.
Repeated sample-specific code is handled by `run_simulation()`; add or remove
samples only in the configuration table.

In [ ]:
suppressPackageStartupMessages({
  library(scDesign3)
  library(SingleCellExperiment)
  library(reticulate)
})

py_require("anndata")

input_dir <- "processed_mf"
output_dir <- "results_simulation"
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

samples <- data.frame(
  input_file = c("sample_7_1.h5ad", "sample_9.h5ad"),
  output_id = c("7_1", "9"),
  n_genes = c(5073, 5090)
)

n_svg <- 100
alphas <- seq(0, 1, by = 0.05)
initial_gp_k <- 100
final_gp_k <- 50
n_cores <- 2
n_extract_cores <- 5

In [ ]:
load_reference <- function(path, n_genes, seed = 1) {
  ad <- import("anndata")$read_h5ad(path)

  if (is.null(ad$raw)) {
    stop("AnnData .raw is required: ", path)
  }

  raw_counts <- as.matrix(t(ad$raw$X))
  gene_names <- as.character(py_to_r(ad$raw$var_names$tolist()))
  spatial <- as.matrix(py_to_r(ad$obsm[["spatial"]]))

  stopifnot(
    length(gene_names) == nrow(raw_counts),
    nrow(spatial) == ncol(raw_counts)
  )

  sce <- SingleCellExperiment(
    assays = list(counts = raw_counts),
    colData = data.frame(
      spatial1 = spatial[, 1],
      spatial2 = spatial[, 2]
    )
  )

  rownames(sce) <- gene_names
  colnames(sce) <- paste0("cell_", seq_len(ncol(sce)))

  expressed_genes <- rownames(sce)[rowSums(counts(sce)) > 0]

  if (length(expressed_genes) < n_genes) {
    stop(
      "Requested ", n_genes, " genes, but only ",
      length(expressed_genes), " have non-zero counts in ", path
    )
  }

  set.seed(seed)
  sce[sample(expressed_genes, n_genes), ]
}

construct_spatial_data <- function(sce) {
  construct_data(
    sce = sce,
    assay_use = "counts",
    celltype = NULL,
    pseudotime = NULL,
    spatial = c("spatial1", "spatial2"),
    other_covariates = NULL,
    corr_by = "1"
  )
}

fit_spatial_marginals <- function(data, gp_k) {
  fit_marginal(
    data = data,
    predictor = "gene",
    mu_formula = sprintf(
      "s(spatial1, spatial2, bs = 'gp', k = %d)",
      gp_k
    ),
    sigma_formula = "1",
    family_use = "nb",
    n_cores = n_cores,
    usebam = FALSE,
    trace = TRUE
  )
}

get_deviance_explained <- function(marginal_list) {
  scores <- vapply(marginal_list, function(model) {
    fit <- model$fit

    if (is.null(fit)) {
      return(NA_real_)
    }

    null_deviance <- fit$null.deviance
    deviance <- fit$deviance

    if (
      is.null(null_deviance) || is.na(null_deviance) ||
      null_deviance <= 0 || is.null(deviance) || is.na(deviance)
    ) {
      return(NA_real_)
    }

    1 - deviance / null_deviance
  }, numeric(1))

  scores[!is.na(scores)]
}

In [ ]:
fit_simulation_model <- function(sce) {
  data <- construct_spatial_data(sce)
  marginal <- fit_spatial_marginals(data, final_gp_k)

  copula <- fit_copula(
    sce = sce,
    assay_use = "counts",
    marginal_list = marginal,
    family_use = "nb",
    copula = "gaussian",
    n_cores = n_cores,
    input_data = data$dat
  )

  parameters <- extract_para(
    sce = sce,
    marginal_list = marginal,
    n_cores = n_extract_cores,
    family_use = "nb",
    new_covariate = data$dat,
    data = data$dat
  )

  list(data = data, copula = copula, parameters = parameters)
}

simulate_signal_gradient <- function(sce, model, seed = 1) {
  set.seed(seed)
  shuffled_rows <- sample(nrow(model$parameters$mean_mat))
  shuffled_mean <- model$parameters$mean_mat[shuffled_rows, ]

  simulated <- lapply(alphas, function(alpha) {
    mean_mat <- (
      alpha * model$parameters$mean_mat +
      (1 - alpha) * shuffled_mean
    )

    count_matrix <- simu_new(
      sce = sce,
      mean_mat = mean_mat,
      sigma_mat = model$parameters$sigma_mat,
      zero_mat = model$parameters$zero_mat,
      quantile_mat = NULL,
      copula_list = model$copula$copula_list,
      n_cores = 1,
      family_use = "nb",
      input_data = model$data$dat,
      new_covariate = model$data$newCovariate,
      important_feature = model$copula$important_feature,
      filtered_gene = model$data$filtered_gene
    )

    rownames(count_matrix) <- paste0(
      rownames(count_matrix), "_", alpha
    )
    count_matrix
  })

  do.call(rbind, simulated)
}

In [ ]:
run_simulation <- function(input_file, output_id, n_genes) {
  message("Processing ", input_file)

  sce <- load_reference(
    file.path(input_dir, input_file),
    n_genes = n_genes
  )

  # The first fit is used only to rank genes by spatial deviance explained.
  initial_data <- construct_spatial_data(sce)
  initial_marginal <- fit_spatial_marginals(
    initial_data,
    initial_gp_k
  )

  deviance <- sort(
    get_deviance_explained(initial_marginal),
    decreasing = TRUE
  )
  selected_genes <- head(names(deviance), n_svg)

  if (length(selected_genes) < n_svg) {
    stop(
      "Only ", length(selected_genes),
      " genes had valid deviance estimates in ", input_file
    )
  }

  svg_sce <- sce[selected_genes, ]
  model <- fit_simulation_model(svg_sce)
  simulated_counts <- simulate_signal_gradient(svg_sce, model)

  write.csv(
    model$data$newCovariate,
    file.path(output_dir, paste0("location_", n_svg, "_", output_id, ".csv"))
  )
  write.csv(
    simulated_counts,
    file.path(output_dir, paste0("counts_", n_svg, "_", output_id, ".csv"))
  )

  message("Finished ", output_id)
  invisible(NULL)
}

In [ ]:
for (i in seq_len(nrow(samples))) {
  run_simulation(
    input_file = samples$input_file[i],
    output_id = samples$output_id[i],
    n_genes = samples$n_genes[i]
  )
}